In [ ]:
import numpy
import pandas

import matplotlib
import matplotlib.pyplot as plt
plt.style.use('../mystyle.mplstyle')

import sbruceana

PATH_TO_SBRUCE = "/Users/triozzi/Analysis/numine/sbruceana/data/"

In [ ]:
FILE = "truth/NuMI_Truth_Dump.txt"

VARS = [
  "index", "pdg", "iscc", "genie_mode", "cc1e0pi", "x", "y", "z",
  "nupx", "nupy", "nupz", "E", "baseline", "xsec",
  "startE", "px", "py", "pz"
]

df = pandas.read_csv(
  f"{PATH_TO_SBRUCE}{FILE}",
  names = VARS,
  delimiter = '\t',
  index_col = False
)

# L / E
df['baseline_over_E'] = (df['baseline']/1.e3) / df['E']

# electron direction
df['p'] = numpy.sqrt(df['px']**2 + df['py']**2 + df['pz']**2)
df['dirx'] = df['px'] / df['p']
df['diry'] = df['py'] / df['p']
df['dirz'] = df['pz'] / df['p']

# neutrino direction
df['nup'] = numpy.sqrt(df['nupx']**2 + df['nupy']**2 + df['nupz']**2)
df['nudirz'] = df['nupz'] / df['nup']

# electron direction with respect to the numi beam axis
NuDirection_NuMI = numpy.array([3.94583e-01, 4.26067e-02, 9.17677e-01])
df['dirnumi'] = (
    df['dirx'] * NuDirection_NuMI[0] +
    df['diry'] * NuDirection_NuMI[1] +
    df['dirz'] * NuDirection_NuMI[2]
)

pot = 4.57524e+20
livetime = 564680
# POT: 4.57524e+20 POT
# livetime: 564680 readouts

#### Breakdown by GENIE CC/NC

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "E"
width = 0.1; bins = numpy.arange(0, 5+width, width)

pot_target = 3e+20
ax = sbruceana.plotting.plot_by_category(ax, df, sbruceana.config.TRUTH_CATEGORIES_CCNC, bins, var, pot_target/pot, True)

# gfx
ax.set(
  title = f'GENIE v3.6.2 sbn1 AR23.20i.00.000\nNuMI MC: {pot_target:.1e} POT',
  xlabel = '$E_{\\nu}$ [GeV]',
  ylabel = f'neutrinos [#]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=10, title='$\\nu$ in AV'); leg.get_title().set_fontsize(11.5)

# scientific formatter
ax.yaxis.set_major_locator(matplotlib.ticker.MultipleLocator(4000))
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14)  # fontsize of the "×10^n" notation

plt.show()
fig.savefig(f"plots/{var}_ccnc.pdf", dpi=300)

#### Breakdown by GENIE interaction mode

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.75), layout='constrained')

# this gives us an idea of the amount of pile up
# in 7.5% of cases, a neutrino in the AV is pile-up with another neutrino
# which interacts sometimes outside of the AV (dirt), and sometimes *within* the AV
# in the latest case, it can be difficult to time the right neutrino...
print('Amount of pileup [%]', 100 * len(df[(df['index'] > 0)]) / len(df))
print('Amount of CC pileup [%]', 100 * len(df[(df['iscc'] == 1) & (df['index'] > 0)]) / len(df[(df['iscc'] == 1)]))
print('Amount of numuCC pileup [%]', 100 * len(df[(df['iscc'] == 1) & (abs(df['pdg']) == 14) & (df['index'] > 0)]) / len(df[(df['iscc'] == 1) & (abs(df['pdg']) == 14)]))
print('Amount of nueCC pileup [%]', 100 * len(df[(df['iscc'] == 1) & (abs(df['pdg']) == 12) & (df['index'] > 0)]) / len(df[(df['iscc'] == 1) & (abs(df['pdg']) == 12)]))
print('Amount of NC pileup [%]', 100 * len(df[(df['iscc'] == 0) & (df['index'] > 0)]) / len(df[(df['iscc'] == 0)]))

var = "index"
width = 1; bins = numpy.arange(0, 5+width, width)

pot_target = 3e+20
ax = sbruceana.plotting.plot_by_category(ax, df, sbruceana.config.TRUTH_CATEGORIES, bins, var, pot_target/pot, True)

ax = sbruceana.plotting.plot_var(
  ax, df[(df.iscc == 1) & (abs(df.pdg) == 12)], bins, var, pot_target/pot, 
  histtype='step', linewidth=2, edgecolor='black', label='$\\nu_e$CC'
)

# gfx
ax.set(
  title = f'GENIE v3.6.2 sbn1 AR23.20i.00.000\nNuMI MC: {pot_target:.1e} POT',
  xlabel = 'GENIE index in spill [#]',
  ylabel = f'neutrinos [#]\n/ {width}',
  xlim   = (bins[0], bins[-1]),
  yscale = 'log'
)
leg = ax.legend(fontsize=10, title='$\\nu$CC in AV'); leg.get_title().set_fontsize(11.5)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "E"
width = 0.1; bins = numpy.arange(0, 5+width, width)

pot_target = 3e+20
ax = sbruceana.plotting.plot_by_category(ax, df, sbruceana.config.TRUTH_CATEGORIES, bins, var, pot_target/pot, True)

ax = sbruceana.plotting.plot_var(
  ax, df[(df.iscc == 1) & (abs(df.pdg) == 12)], bins, var, pot_target/pot, 
  histtype='step', linewidth=2, edgecolor='black', label='$\\nu_e$CC'
)

# gfx
ax.set(
  title = f'GENIE v3.6.2 sbn1 AR23.20i.00.000\nNuMI MC: {pot_target:.1e} POT',
  xlabel = '$E_{\\nu}$ [GeV]',
  ylabel = f'neutrinos [#]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
  ylim   = (0, 1e4)
)
leg = ax.legend(fontsize=10, title='$\\nu$CC in AV'); leg.get_title().set_fontsize(11.5)

# scientific formatter
ax.yaxis.set_major_locator(matplotlib.ticker.MultipleLocator(2000))
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14)  # fontsize of the "×10^n" notation

plt.show()
fig.savefig(f"plots/{var}_CC.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.75), layout='constrained')

var = "baseline"
width = 5; bins = numpy.arange(525, 825+width, width)

pot_target = 3e+20
ax = sbruceana.plotting.plot_by_category(ax, df, sbruceana.config.TRUTH_CATEGORIES, bins, var, pot_target/pot, True)

ax = sbruceana.plotting.plot_var(
  ax, df[(df.iscc == 1) & (abs(df.pdg) == 12)], bins, var, pot_target/pot, 
  histtype='step', linewidth=2, edgecolor='black', label='$\\nu_e$CC'
)

# gfx
ax.set(
  title = f'GENIE v3.6.2 sbn1 AR23.20i.00.000\nNuMI MC: {pot_target:.1e} POT',
  xlabel = 'baseline [m]',
  ylabel = f'neutrinos [#]\n/ {width} m',
  xlim   = (bins[0], bins[-1]),
  # yscale = 'log'
)
leg = ax.legend(fontsize=10, loc='upper left', title='$\\nu$CC in AV'); leg.get_title().set_fontsize(11.5)

# scientific formatter
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14)  # fontsize of the "×10^n" notation

plt.show()
fig.savefig(f"plots/{var}_CC.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.75), layout='constrained')

var = "baseline_over_E"
width = 0.05; bins = numpy.arange(0, 3+width, width)

pot_target = 3e+20
ax = sbruceana.plotting.plot_by_category(ax, df, sbruceana.config.TRUTH_CATEGORIES, bins, var, pot_target/pot, True)

ax = sbruceana.plotting.plot_var(
  ax, df[(df.iscc == 1) & (abs(df.pdg) == 12)], bins, var, pot_target/pot, 
  histtype='step', linewidth=2, edgecolor='black', label='$\\nu_e$CC'
)

# gfx
ax.set(
  title = f'GENIE v3.6.2 sbn1 AR23.20i.00.000\nNuMI MC: {pot_target:.1e} POT',
  xlabel = '$L$/$E_{\\nu}$ [km/GeV]',
  ylabel = f'neutrinos [#]\n/ {width} km/GeV',
  xlim   = (bins[0], bins[-1]),
  # yscale = 'log'
)
leg = ax.legend(fontsize=10, loc='upper right', title='$\\nu$CC in AV'); leg.get_title().set_fontsize(11.5)

# scientific formatter
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14)  # fontsize of the "×10^n" notation

plt.show()
fig.savefig(f"plots/{var}_CC.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.75), layout='constrained')

var = "E"
width = 0.1; bins = numpy.arange(0, 5+width, width)

pot_target = 3e+20
ax = sbruceana.plotting.plot_by_category(ax, df[abs(df.pdg) == 14], sbruceana.config.TRUTH_CATEGORIES, bins, var, pot_target/pot, True)
# ax.hist(df[var], bins=bins)

# gfx
ax.set(
  title = f'GENIE v3.6.2 sbn1 AR23.20i.00.000\nNuMI MC: {pot_target:.1e} POT',
  xlabel = '$E_{\\nu}$ [GeV]',
  ylabel = f'neutrinos [#]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=10, title='$\\nu_{\\mu}$CC in AV'); leg.get_title().set_fontsize(11.5)

# scientific formatter
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14)  # fontsize of the "×10^n" notation

plt.show()
fig.savefig(f"plots/{var}_numu_CC.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.75), layout='constrained')

varX = "E"
widthX = 0.1; binsX = numpy.arange(0, 5+widthX, widthX)

varY = "baseline"
widthY = 5; binsY = numpy.arange(525, 825+widthY, widthY)

h = ax.hist2d(
  df[abs(df.pdg) == 14][varX],
  df[abs(df.pdg) == 14][varY],
  bins = (binsX, binsY),
  cmap = 'magma_r',
  # norm=matplotlib.colors.LogNorm()
)

# inset colorbar
cax = ax.inset_axes([0.635, 0.3, 0.25, 0.05])  # [x, y, w, h]
fig.colorbar(h[3], cax=cax, orientation='horizontal')

# gfx
ax.set(
  title = f'GENIE v3.6.2 sbn1 AR23.20i.00.000\nNuMI MC: {pot_target:.1e} POT',
  xlabel = '$E_{\\nu}$ [GeV] / ' + f'{widthX} GeV',
  ylabel = f'baseline [m] / {widthY} m',
  xlim   = (bins[0], bins[-1]),
)

ax.text(3., 550, '$\\nu_{\\mu}$CC in AV', color='black')

plt.show()
fig.savefig(f"plots/{varX}_{varY}_numu_CC.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.75), layout='constrained')

varX = "E"
widthX = 0.1; binsX = numpy.arange(0, 5+widthX, widthX)

varY = "baseline"
widthY = 5; binsY = numpy.arange(525, 825+widthY, widthY)

h = ax.hist2d(
  df[abs(df.pdg) == 12][varX],
  df[abs(df.pdg) == 12][varY],
  bins = (binsX, binsY),
  cmap = 'magma_r',
)

# inset colorbar
cax = ax.inset_axes([0.635, 0.3, 0.25, 0.05])  # [x, y, w, h]
fig.colorbar(h[3], cax=cax, orientation='horizontal')

# gfx
ax.set(
  title = f'GENIE v3.6.2 sbn1 AR23.20i.00.000\nNuMI MC: {pot_target:.1e} POT',
  xlabel = '$E_{\\nu}$ [GeV] / ' + f'{widthX} GeV',
  ylabel = f'baseline [m] / {widthY} m',
  xlim   = (bins[0], bins[-1]),
)

ax.text(3., 550, '$\\nu_{e}$CC in AV', color='black')

plt.show()
fig.savefig(f"plots/{varX}_{varY}_nue_CC.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.75), layout='constrained')

var = "E"
width = 0.20; bins = numpy.arange(0, 5+width, width)

pot_target = 3e+20
ax = sbruceana.plotting.plot_by_category(ax, df[abs(df.pdg) == 12], sbruceana.config.TRUTH_CATEGORIES, bins, var, pot_target/pot, True)

# ax.hist(df[df.cc1e0pi == 1][var], bins=bins, histtype='step', linewidth=2, edgecolor='black', label='1eNp0π')


# gfx
ax.set(
  title = f'GENIE v3.6.2 sbn1 AR23.20i.00.000\nNuMI MC: {pot_target:.1e} POT',
  xlabel = '$E_{\\nu}$ [GeV]',
  ylabel = f'neutrinos [#]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=10, title='$\\nu_e$CC in AV'); leg.get_title().set_fontsize(11.5)

# scientific formatter
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14)  # fontsize of the "×10^n" notation

plt.show()
fig.savefig(f"plots/{var}_nue_CC.pdf", dpi=300)

#### Breakdown by GENIE interaction mode -- 1eNp0π

In [ ]:
100 * len(df[(df.cc1e0pi == 1) & (df.pdg < 0)]) / len(df[df.cc1e0pi == 1])

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.75), layout='constrained')

var = "E"
width = 0.20; bins = numpy.arange(0, 5+width, width)

pot_target = 3e+20
ax = sbruceana.plotting.plot_by_category(ax, df[df.cc1e0pi == 1], sbruceana.config.TRUTH_CATEGORIES, bins, var, pot_target/pot, True)

ax = sbruceana.plotting.plot_var(
  ax, df[(df.cc1e0pi == 1) & (df.pdg < 0)], bins, var, pot_target/pot, 
  histtype='step', linewidth=1.5, edgecolor='black', label='$\\overline{\\nu}$ 1eNp0π'
)

# gfx
ax.set(
  title = f'GENIE v3.6.2 sbn1 AR23.20i.00.000\nNuMI MC: {pot_target:.1e} POT',
  xlabel = '$E_{\\nu}$ [GeV]',
  ylabel = f'neutrinos [#]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=10, title='1eNp0π'); leg.get_title().set_fontsize(11.5)

# scientific formatter
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14)  # fontsize of the "×10^n" notation

plt.show()
fig.savefig(f"plots/{var}_1eNp0π.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.75), layout='constrained')

var = "baseline_over_E"
width = 0.1; bins = numpy.arange(0, 3+width, width)

pot_target = 3e+20
ax = sbruceana.plotting.plot_by_category(ax, df[df.cc1e0pi == 1], sbruceana.config.TRUTH_CATEGORIES, bins, var, pot_target/pot, True)

ax = sbruceana.plotting.plot_var(
  ax, df[(df.cc1e0pi == 1) & (df.pdg < 0)], bins, var, pot_target/pot, 
  histtype='step', linewidth=1.5, edgecolor='black', label='$\\overline{\\nu}$ 1eNp0π'
)

# gfx
ax.set(
  title = f'GENIE v3.6.2 sbn1 AR23.20i.00.000\nNuMI MC: {pot_target:.1e} POT',
  xlabel = '$L$/$E_{\\nu}$ [km/GeV]',
  ylabel = f'neutrinos [#]\n/ {width} km/GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=10, title='1eNp0π'); leg.get_title().set_fontsize(11.5)

# scientific formatter
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14)  # fontsize of the "×10^n" notation

plt.show()
fig.savefig(f"plots/{var}_1eNp0π.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.75), layout='constrained')

var = "startE"
width = 0.2; bins = numpy.arange(0.2, 3+width, width)

pot_target = 3e+20
ax = sbruceana.plotting.plot_by_category(ax, df[df.cc1e0pi == 1], sbruceana.config.TRUTH_CATEGORIES, bins, var, pot_target/pot, True)

ax = sbruceana.plotting.plot_var(
  ax, df[(df.cc1e0pi == 1) & (df.pdg < 0)], bins, var, pot_target/pot, 
  histtype='step', linewidth=1.5, edgecolor='black', label='$\\overline{\\nu}$ 1eNp0π'
)

# gfx
ax.set(
  title = f'GENIE v3.6.2 sbn1 AR23.20i.00.000\nNuMI MC: {pot_target:.1e} POT',
  xlabel = '$E_{e^{\\pm}}$ [GeV]',
  ylabel = f'neutrinos [#]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=10, title='1eNp0π'); leg.get_title().set_fontsize(11.5)

# scientific formatter
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14)  # fontsize of the "×10^n" notation

plt.show()
fig.savefig(f"plots/{var}_1eNp0π.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.75), layout='constrained')

var = "dirz"
width = 0.1; bins = numpy.arange(-1, 1+width, width)

pot_target = 3e+20
ax = sbruceana.plotting.plot_by_category(ax, df[df.cc1e0pi == 1], sbruceana.config.TRUTH_CATEGORIES, bins, var, pot_target/pot, True)

ax = sbruceana.plotting.plot_var(
  ax, df[(df.cc1e0pi == 1) & (df.pdg < 0)], bins, var, pot_target/pot, 
  histtype='step', linewidth=1.5, edgecolor='black', label='$\\overline{\\nu}$ 1eNp0π'
)

# gfx
ax.set(
  title = f'GENIE v3.6.2 sbn1 AR23.20i.00.000\nNuMI MC: {pot_target:.1e} POT',
  xlabel = '$e^{\\pm}$ cos($\\theta_z$)',
  ylabel = f'neutrinos [#]\n/ {width}',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=10, title='1eNp0π'); leg.get_title().set_fontsize(11.5)

# scientific formatter
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14)  # fontsize of the "×10^n" notation

plt.show()
fig.savefig(f"plots/{var}_1eNp0π.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.75), layout='constrained')

var = "dirnumi"
width = 0.1; bins = numpy.arange(-1, 1+width, width)

pot_target = 3e+20
ax = sbruceana.plotting.plot_by_category(ax, df[df.cc1e0pi == 1], sbruceana.config.TRUTH_CATEGORIES, bins, var, pot_target/pot, True)

ax = sbruceana.plotting.plot_var(
  ax, df[(df.cc1e0pi == 1) & (df.pdg < 0)], bins, var, pot_target/pot, 
  histtype='step', linewidth=1.5, edgecolor='black', label='$\\overline{\\nu}$ 1eNp0π'
)

# gfx
ax.set(
  title = f'GENIE v3.6.2 sbn1 AR23.20i.00.000\nNuMI MC: {pot_target:.1e} POT',
  xlabel = '$e^{\\pm}$ cos($\\theta_\\mathrm{NuMI}$)',
  ylabel = f'neutrinos [#]\n/ {width}',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=10, title='1eNp0π'); leg.get_title().set_fontsize(11.5)

# scientific formatter
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14)  # fontsize of the "×10^n" notation

plt.show()
fig.savefig(f"plots/{var}_1eNp0π.pdf", dpi=300)

### 1μNp0π

In [ ]:
FILE = "truth/NuMI_NuMu_Truth_Dump.txt"

VARS = [
  "index", "pdg", "iscc", "genie_mode", "cc1mu0pi", "x", "y", "z",
  "nupx", "nupy", "nupz", "E", "baseline", "xsec",
  "startendE", "length", "px", "py", "pz"
]

df = pandas.read_csv(
  f"{PATH_TO_SBRUCE}{FILE}",
  names = VARS,
  delimiter = '\t',
  index_col = False
)

# L / E
df['baseline_over_E'] = (df['baseline']/1.e3) / df['E']

# electron direction
df['p'] = numpy.sqrt(df['px']**2 + df['py']**2 + df['pz']**2)
df['dirx'] = df['px'] / df['p']
df['diry'] = df['py'] / df['p']
df['dirz'] = df['pz'] / df['p']

# neutrino direction
df['nup'] = numpy.sqrt(df['nupx']**2 + df['nupy']**2 + df['nupz']**2)
df['nudirz'] = df['nupz'] / df['nup']

# electron direction with respect to the numi beam axis
NuDirection_NuMI = numpy.array([3.94583e-01, 4.26067e-02, 9.17677e-01])
df['dirnumi'] = (
    df['dirx'] * NuDirection_NuMI[0] +
    df['diry'] * NuDirection_NuMI[1] +
    df['dirz'] * NuDirection_NuMI[2]
)

pot = 4.57524e+20
livetime = 564680
# POT: 4.57524e+20 POT
# livetime: 564680 readouts

#### Breakdown by GENIE interaction mode -- 1eNp0π

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.75), layout='constrained')

var = "E"
width = 0.1; bins = numpy.arange(0, 3.5+width, width)

pot_target = 3e+20
ax = sbruceana.plotting.plot_by_category(ax, df[df.cc1mu0pi == 1], sbruceana.config.TRUTH_CATEGORIES, bins, var, pot_target/pot, True)

# gfx
ax.set(
  title = f'GENIE v3.6.2 sbn1 AR23.20i.00.000\nNuMI MC: {pot_target:.1e} POT',
  xlabel = '$E_{\\nu}$ [GeV]',
  ylabel = f'neutrinos [#]\n/ {width} GeV',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=10, title='1μNp0π'); leg.get_title().set_fontsize(11.5)

# scientific formatter
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14)  # fontsize of the "×10^n" notation

plt.show()
fig.savefig(f"plots/{var}_1muNp0π.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.75), layout='constrained')

var = "length"
width = 10; bins = numpy.arange(50, 500+width, width)

pot_target = 3e+20
ax = sbruceana.plotting.plot_by_category(ax, df[df.cc1mu0pi == 1], sbruceana.config.TRUTH_CATEGORIES, bins, var, pot_target/pot, True)

# gfx
ax.set(
  title = f'GENIE v3.6.2 sbn1 AR23.20i.00.000\nNuMI MC: {pot_target:.1e} POT',
  xlabel = '$L_{\\mu}$ [m]',
  ylabel = f'neutrinos [#]\n/ {width} m',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=10, title='1μNp0π'); leg.get_title().set_fontsize(11.5)

# scientific formatter
ax.yaxis.set_major_locator(matplotlib.ticker.MultipleLocator(0.2e3))
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14)  # fontsize of the "×10^n" notation

plt.show()
fig.savefig(f"plots/{var}_1muNp0π.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.75), layout='constrained')

var = "dirz"
width = 0.05; bins = numpy.arange(-1, 1+width, width)

pot_target = 3e+20
ax = sbruceana.plotting.plot_by_category(ax, df[df.cc1mu0pi == 1], sbruceana.config.TRUTH_CATEGORIES, bins, var, pot_target/pot, True)

# gfx
ax.set(
  title = f'GENIE v3.6.2 sbn1 AR23.20i.00.000\nNuMI MC: {pot_target:.1e} POT',
  xlabel = '$\\mu^{\\pm}$ cos($\\theta_z$)',
  ylabel = f'neutrinos [#]\n/ {width} m',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=10, title='1μNp0π'); leg.get_title().set_fontsize(11.5)

# scientific formatter
ax.yaxis.set_major_locator(matplotlib.ticker.MultipleLocator(1.0e3))
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14)  # fontsize of the "×10^n" notation

plt.show()
fig.savefig(f"plots/{var}_1muNp0π.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.75), layout='constrained')

var = "dirnumi"
width = 0.05; bins = numpy.arange(-1, 1+width, width)

pot_target = 3e+20
ax = sbruceana.plotting.plot_by_category(ax, df[df.cc1mu0pi == 1], sbruceana.config.TRUTH_CATEGORIES, bins, var, pot_target/pot, True)

# gfx
ax.set(
  title = f'GENIE v3.6.2 sbn1 AR23.20i.00.000\nNuMI MC: {pot_target:.1e} POT',
  xlabel = '$\\mu^{\\pm}$ cos($\\theta_\\mathrm{NuMI}$)',
  ylabel = f'neutrinos [#]\n/ {width} m',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=10, title='1μNp0π'); leg.get_title().set_fontsize(11.5)

# scientific formatter
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.offsetText.set_fontsize(14)  # fontsize of the "×10^n" notation

plt.show()
fig.savefig(f"plots/{var}_1muNp0π.pdf", dpi=300)